# Protocolo de Validação — Caldeira de Geração de Vapor (Arquitetura Final 1B)

Este caderno executa de forma automatizada os **7 ensaios do Protocolo de Validação** sobre a arquitetura consolidada em `Caldeira_Completa.slx`:

* **Malha 1 (Vapor):** `GPC SISO` ($T_s = 0{,}5\text{ s}$, $N_1 = 1$, $N_2 = 20$, $N_u = 2$, $\lambda_v = 1{,}2$, $F_{g,\text{ref}} \in [0{,}0;\; 0{,}48]$, $\Delta u \in [-0{,}30;\; +0{,}30]$).
* **Malha 2 (Queima 1B):**
  * **Gás:** `I-P (b = 0)` com polos em $(-8,\; -12)$ ($K_{p,g} = 0{,}50$, $K_{i,g} = 4{,}80$) e *anti-windup* contínuo (*back-calculation*, $K_b = 50$).
  * **Ar:** `PI LGR` com polos em $(-10,\; -16)$ ($K_{p,r} = 4{,}20$, $K_{i,r} = 32{,}00$) rastreando $R \cdot F_g$ ($R = 2{,}0$), somado ao **Feedforward realizável** $G_{\text{FF,ar}}(s) = 4R - \frac{2R}{0{,}1s+1}$ a partir de $a_g$.
* **Malha 3 (Nível):** `GPC SISO` ($T_{s,L} = 1{,}0\text{ s}$, $N_1 = 1$, $N_2 = 30$, $N_u = 1$, $\lambda_L = 150$, $\Delta V_a \in [-0{,}05;\; +0{,}05]$) com **Feedforward Dinâmico Exato externo** $G_{\text{FF,L}}(s) = -0{,}4 + \frac{0{,}5}{5s+1}$, deslocamento dinâmico dos limites $[0 - v_{\text{ff}};\; 1 - v_{\text{ff}}]$ e proteção de factibilidade.

> Clique em **Ambiente de execução → Executar tudo** para rodar os 7 testes e gerar a tabela consolidada ao final.

In [ ]:
import numpy as np
import scipy.signal as signal
import scipy.optimize as opt
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.35
plt.rcParams["font.size"] = 9.5

print("Bibliotecas carregadas. Pronto para iniciar a bateria de validação.")

## 1. Núcleo de Simulação (`ControladorGPC` + Simulador Multitaxa RK4 + Métricas)

* `ControladorGPC`: implementação fiel ao `ControladorPreditivo.m` e `ControladorPreditivoDin.m`, incluindo a proteção de factibilidade nos limites dinâmicos e contagem exata de eventuais falhas no otimizador (`exitflag != 1`).
* `simular_protocolo`: integra a planta contínua em passo $dt = 2\text{ ms}$ (RK4), executa o GPC de vapor a cada $0{,}5\text{ s}$ e o GPC de nível a cada $1{,}0\text{ s}$, permitindo variar parâmetros reais da planta (Teste 5) e injetar ruído de medição (Teste 6).
* `calcular_metricas_ensaio`: extrai $t_{5\%}$, $t_{2\%}$, $OS\%$, erro em regime, `IAE`, `ITAE`, $\min(F_r/F_g)$, desvio máximo de razão, tempo de recuperação $\pm 2\%$, variação total $TV = \sum |u(k)-u(k-1)|$ e contagem de falhas no QP.

In [ ]:
def discretizar_planta(num_p, den_p, Ts):
    num_d, den_d, _ = signal.cont2discrete((np.asarray(num_p, float), np.asarray(den_p, float)), Ts, method="zoh")
    den = den_d.ravel()
    num = num_d.ravel()
    if len(num) < len(den):
        num = np.pad(num, (len(den) - len(num), 0))
    return den / den[0], num[1:] / den[0]


class ControladorGPC:
    def __init__(self, A, B, d, N1, N2, Nu, lam, u0, umin, umax, dumin, dumax, ymin, ymax):
        self.A = np.asarray(A, dtype=float)
        self.B = np.asarray(B, dtype=float)
        self.d = int(d)
        self.N1 = int(N1)
        self.N2 = int(N2)
        self.Nu = int(Nu)
        self.lam = float(lam)
        self.Ny = self.N2 - self.N1 + 1

        self.umin = float(umin)
        self.umax = float(umax)
        self.dumin = float(dumin)
        self.dumax = float(dumax)
        self.ymin = float(ymin)
        self.ymax = float(ymax)

        grau_A = len(self.A) - 1
        A_til = np.convolve(self.A, np.array([1.0, -1.0]))

        E_full = np.zeros((self.N2, self.N2))
        F_full = np.zeros((self.N2, grau_A + 1))
        E_full[0, 0] = 1.0
        F_full[0, :] = -A_til[1:]

        for passo in range(self.N2 - 1):
            termo = F_full[passo, 0]
            E_full[passo + 1, : passo + 1] = E_full[passo, : passo + 1]
            E_full[passo + 1, passo + 1] = termo
            temp_F = np.concatenate([F_full[passo, :], [0.0]]) - termo * A_til
            F_full[passo + 1, :] = temp_F[1:]

        n_past = len(self.B) - 1 + self.d
        G_full = np.zeros((self.N2, self.N2))
        H_full = np.zeros((self.N2, n_past))

        for idx in range(self.N2):
            j = idx + 1
            Mj = np.convolve(E_full[idx, :j], self.B)
            n_fut = max(j - self.d, 0)
            if n_fut > 0:
                G_full[idx, :n_fut] = Mj[:n_fut][::-1]
            past = Mj[n_fut:]
            if n_past > 0 and len(past) > 0:
                offset = max(self.d - j, 0)
                H_full[idx, offset : offset + len(past)] = past

        self.G = G_full[self.N1 - 1 : self.N2, : self.Nu]
        self.H = H_full[self.N1 - 1 : self.N2, :]
        self.F = F_full[self.N1 - 1 : self.N2, :]

        Qu = self.lam * np.eye(self.Nu)
        self.Hqp = 2.0 * (self.G.T @ self.G + Qu)
        T = np.tril(np.ones((self.Nu, self.Nu)))
        self.A_qp = np.vstack([T, -T, self.G, -self.G])
        self.bounds = [(self.dumin, self.dumax)] * self.Nu

        self.n_mem_y = len(self.A)
        self.n_mem_du = max(n_past, 1)
        self.u0 = float(u0)
        self.reset()

    def reset(self, u0_override=None):
        self.y_mem = np.zeros(self.n_mem_y)
        self.du_mem = np.zeros(self.n_mem_du)
        self.u_anterior = self.u0 if u0_override is None else float(u0_override)
        self.primeira_execucao = True
        self.n_fail_qp = 0

    def resolver_qp(self, fqp, A_ineq, b_ineq):
        Hqp = self.Hqp
        x0 = np.zeros(self.Nu)
        x0[0] = np.clip(0.0, self.dumin, self.dumax)
        return opt.minimize(
            fun=lambda x: 0.5 * float(x @ Hqp @ x) + float(fqp @ x),
            x0=x0,
            jac=lambda x: Hqp @ x + fqp,
            bounds=self.bounds,
            constraints={
                "type": "ineq",
                "fun": lambda x: b_ineq - A_ineq @ x,
                "jac": lambda x: -A_ineq,
            },
            method="SLSQP",
            options={"ftol": 1e-10, "maxiter": 250},
        )

    def step(self, r, y, umin_override=None, umax_override=None):
        if self.primeira_execucao:
            self.y_mem[:] = y
            self.primeira_execucao = False

        self.y_mem = np.roll(self.y_mem, 1)
        self.y_mem[0] = y

        w = r * np.ones(self.Ny)
        if self.H.shape[1] > 0:
            f = self.F @ self.y_mem + self.H @ self.du_mem[: self.H.shape[1]]
        else:
            f = self.F @ self.y_mem

        fqp = -2.0 * (self.G.T @ (w - f))
        u_ant = self.u_anterior

        umin_d = self.umin if umin_override is None else float(umin_override)
        umax_d = self.umax if umax_override is None else float(umax_override)
        umin_ef = min(umin_d, u_ant + self.dumax)
        umax_ef = max(umax_d, u_ant + self.dumin)

        b_qp = np.concatenate([
            (umax_ef - u_ant) * np.ones(self.Nu),
            (u_ant - umin_ef) * np.ones(self.Nu),
            self.ymax * np.ones(self.Ny) - f,
            -self.ymin * np.ones(self.Ny) + f,
        ])

        res = self.resolver_qp(fqp, self.A_qp, b_qp)
        if not res.success:
            res2 = self.resolver_qp(fqp, self.A_qp[: 2 * self.Nu, :], b_qp[: 2 * self.Nu])
            if not res2.success:
                self.n_fail_qp += 1
                du = 0.0
            else:
                du = float(res2.x[0])
        else:
            du = float(res.x[0])

        u = u_ant + du
        self.du_mem = np.roll(self.du_mem, 1)
        self.du_mem[0] = du
        self.u_anterior = u
        return u, du


# Modelos nominais dos controladores (fixos em todos os testes)
R_NOM = 2.0
KPG, KIG, BG = 0.50, 4.80, 0.0
KPR, KIR, BR = 4.20, 32.00, 1.0
KB_AW = 50.0

den_tg_nom = np.array([1.0, 10.0 + 20.0 * KPG, 20.0 * KIG])
num_gv_nom = np.array([10.0 * 20.0 * KIG])
den_gv_nom = np.convolve(np.convolve([5.0, 1.0], [3.0, 1.0]), den_tg_nom)
AV_NOM, BV_NOM = discretizar_planta(num_gv_nom, den_gv_nom, 0.5)
AL_NOM, BL_NOM = discretizar_planta([5.0], [50.0, 1.0], 1.0)


def simular_protocolo(
    t_final=160.0,
    r_v_fn=lambda t: 0.0,
    r_L_fn=lambda t: 2.0,
    dg_fn=lambda t: 0.0,
    dVa_fn=lambda t: 0.0,
    Fv_0=0.0,
    L_0=2.0,
    tau_g_real=0.10,
    tau_r_real=0.20,
    K_cal_real=10.0,
    tau_sw_real=20.0,
    noise_std=None,
    tau_f_meas=0.0,
    seed=42,
):
    dt = 0.002
    n_steps = int(round(t_final / dt)) + 1
    Ts_v, Ts_L = 0.5, 1.0
    sub_v = int(round(Ts_v / dt))
    sub_L = int(round(Ts_L / dt))
    passo_log = 10  # 20 ms

    gpc_vap = ControladorGPC(AV_NOM, BV_NOM, 0, 1, 20, 2, 1.2, 0.0, 0.0, 0.48, -0.30, 0.30, 0.0, 200.0)
    gpc_niv = ControladorGPC(AL_NOM, BL_NOM, 0, 1, 30, 1, 150.0, 0.40, 0.0, 1.0, -0.05, 0.05, -10.0, 50.0)

    # Equilíbrio inicial exato para (Fv_0, L_0) na planta real
    Fg_0 = Fv_0 / K_cal_real
    ag_0 = Fg_0 / 2.0
    Fr_0 = R_NOM * Fg_0
    ar_0 = Fr_0
    Ig_0 = ag_0 + KPG * Fg_0
    Ir_0 = ar_0 - (4.0 * R_NOM * ag_0 - R_NOM * Fg_0)
    Va_0 = (L_0 + 0.5 * Fv_0) / 5.0
    vffe_0 = 0.5 * Fv_0
    Va_gpc_0 = Va_0 - (-0.4 * Fv_0 + vffe_0)
    zd_0 = 0.5 * (1.0 + 0.2 * tau_sw_real) * Fv_0

    gpc_vap.reset(u0_override=Fg_0)
    gpc_niv.reset(u0_override=Va_gpc_0)

    Fg_v = Fg_0
    x1 = Fv_0
    Fv = Fv_0
    Ig = Ig_0
    z_ff_g = Fg_0
    Fr = Fr_0
    Ir = Ir_0
    L = L_0
    zd = zd_0
    vff_e = vffe_0

    Fg_ref = Fg_0
    Va_gpc = Va_gpc_0

    rng = np.random.default_rng(seed)
    n_log = (n_steps - 1) // passo_log + 1
    t_log = np.zeros(n_log)
    r_v_log = np.zeros(n_log)
    Fv_log = np.zeros(n_log)
    Fg_ref_log = np.zeros(n_log)
    Fg_log = np.zeros(n_log)
    Fr_log = np.zeros(n_log)
    ag_log = np.zeros(n_log)
    ar_log = np.zeros(n_log)
    r_L_log = np.zeros(n_log)
    L_log = np.zeros(n_log)
    Va_log = np.zeros(n_log)
    vff_log = np.zeros(n_log)
    li = 0

    # Ruído mantido em janela de 20 ms para compatibilidade numérica com RK4
    n_Fv = n_Fg = n_Fr = n_L = 0.0
    alpha_f = (1.0 - np.exp(-dt / tau_f_meas)) if tau_f_meas > 0.0 else 1.0
    Fg_filt, Fr_filt, Fv_filt, L_filt = Fg_0, Fr_0, Fv_0, L_0

    def derivs(s_Fgv, s_x1, s_Fv, s_Ig, s_zffg, s_Fr, s_Ir, s_L, s_zd, s_vffe, c_Fg_ref, c_Va_gpc, dg, dVa, w_Fg, w_Fr):
        Fg_tot = s_Fgv + dg
        Fg_m = Fg_tot + w_Fg
        Fr_m = s_Fr + w_Fr

        # Malha 2: I-P Gás (b = 0) com back-calculation
        eg = c_Fg_ref - Fg_m
        ag_raw = -KPG * Fg_m + s_Ig
        ag = 1.0 if ag_raw > 1.0 else (0.0 if ag_raw < 0.0 else ag_raw)
        dIg = KIG * eg + KB_AW * (ag - ag_raw)

        # FF realizável de ag -> ar: 4R*ag - 2R/(0.1s+1)*ag
        ar_ff = 4.0 * R_NOM * ag - R_NOM * s_zffg
        dzffg = (2.0 * ag - s_zffg) / 0.1

        # PI LGR Ar + FF + back-calculation
        er = R_NOM * Fg_m - Fr_m
        ar_raw = KPR * er + s_Ir + ar_ff
        ar = 1.0 if ar_raw > 1.0 else (0.0 if ar_raw < 0.0 else ar_raw)
        dIr = KIR * er + KB_AW * (ar - ar_raw)

        dFgv = (2.0 * ag - s_Fgv) / tau_g_real
        dx1 = (K_cal_real * Fg_tot - s_x1) / 5.0
        dFv = (s_x1 - s_Fv) / 3.0
        dFr = (ar - s_Fr) / tau_r_real

        # Malha 3: FF Dinâmico Exato + Tubulão
        v_ff = -0.4 * s_Fv + s_vffe
        Va_tot = c_Va_gpc + v_ff
        Va_tot = 1.0 if Va_tot > 1.0 else (0.0 if Va_tot < 0.0 else Va_tot)
        dzd = (0.5 * (1.0 + 0.2 * tau_sw_real) * s_Fv - s_zd) / 5.0
        dL = (5.0 * (Va_tot + dVa) + 0.1 * tau_sw_real * s_Fv - s_zd - s_L) / 50.0
        dvffe = (0.5 * s_Fv - s_vffe) / 5.0

        return dFgv, dx1, dFv, dIg, dzffg, dFr, dIr, dL, dzd, dvffe, ag, ar, Va_tot, Fg_tot, v_ff

    for i in range(n_steps):
        tk = i * dt
        rv = float(r_v_fn(tk))
        rL = float(r_L_fn(tk))
        dg = float(dg_fn(tk))
        dVa = float(dVa_fn(tk))

        if noise_std is not None and (i % passo_log == 0):
            n_Fv = float(rng.normal(0.0, noise_std.get("Fv", 0.0)))
            n_Fg = float(rng.normal(0.0, noise_std.get("Fg", 0.0)))
            n_Fr = float(rng.normal(0.0, noise_std.get("Fr", 0.0)))
            n_L = float(rng.normal(0.0, noise_std.get("L", 0.0)))

        if tau_f_meas > 0.0:
            Fg_filt += alpha_f * ((Fg_v + dg + n_Fg) - Fg_filt)
            Fr_filt += alpha_f * ((Fr + n_Fr) - Fr_filt)
            Fv_filt += alpha_f * ((Fv + n_Fv) - Fv_filt)
            L_filt  += alpha_f * ((L + n_L) - L_filt)
            w_Fg_eff = Fg_filt - (Fg_v + dg)
            w_Fr_eff = Fr_filt - Fr
            Fv_meas = Fv_filt
            L_meas = L_filt
        else:
            w_Fg_eff = n_Fg
            w_Fr_eff = n_Fr
            Fv_meas = Fv + n_Fv
            L_meas = L + n_L

        if i % sub_v == 0:
            Fg_ref, _ = gpc_vap.step(rv, Fv_meas)

        if i % sub_L == 0:
            v_ff_k = -0.4 * Fv_meas + vff_e
            Va_gpc, _ = gpc_niv.step(rL, L_meas, umin_override=0.0 - v_ff_k, umax_override=1.0 - v_ff_k)

        if i % passo_log == 0:
            _, _, _, _, _, _, _, _, _, _, ag_v, ar_v, Va_t, Fg_t, vff_t = derivs(
                Fg_v, x1, Fv, Ig, z_ff_g, Fr, Ir, L, zd, vff_e, Fg_ref, Va_gpc, dg, dVa, w_Fg_eff, w_Fr_eff
            )
            t_log[li] = tk
            r_v_log[li] = rv
            Fv_log[li] = Fv
            Fg_ref_log[li] = Fg_ref
            Fg_log[li] = Fg_t
            Fr_log[li] = Fr
            ag_log[li] = ag_v
            ar_log[li] = ar_v
            r_L_log[li] = rL
            L_log[li] = L
            Va_log[li] = Va_t
            vff_log[li] = vff_t
            li += 1

        if i < n_steps - 1:
            tm = tk + 0.5 * dt
            tn = tk + dt
            dgm, dVm = float(dg_fn(tm)), float(dVa_fn(tm))
            dgn, dVn = float(dg_fn(tn)), float(dVa_fn(tn))

            k1 = derivs(Fg_v, x1, Fv, Ig, z_ff_g, Fr, Ir, L, zd, vff_e, Fg_ref, Va_gpc, dg, dVa, n_Fg, n_Fr)
            k2 = derivs(
                Fg_v + 0.5*dt*k1[0], x1 + 0.5*dt*k1[1], Fv + 0.5*dt*k1[2], Ig + 0.5*dt*k1[3], z_ff_g + 0.5*dt*k1[4],
                Fr + 0.5*dt*k1[5], Ir + 0.5*dt*k1[6], L + 0.5*dt*k1[7], zd + 0.5*dt*k1[8], vff_e + 0.5*dt*k1[9],
                Fg_ref, Va_gpc, dgm, dVm, n_Fg, n_Fr
            )
            k3 = derivs(
                Fg_v + 0.5*dt*k2[0], x1 + 0.5*dt*k2[1], Fv + 0.5*dt*k2[2], Ig + 0.5*dt*k2[3], z_ff_g + 0.5*dt*k2[4],
                Fr + 0.5*dt*k2[5], Ir + 0.5*dt*k2[6], L + 0.5*dt*k2[7], zd + 0.5*dt*k2[8], vff_e + 0.5*dt*k2[9],
                Fg_ref, Va_gpc, dgm, dVm, n_Fg, n_Fr
            )
            k4 = derivs(
                Fg_v + dt*k3[0], x1 + dt*k3[1], Fv + dt*k3[2], Ig + dt*k3[3], z_ff_g + dt*k3[4],
                Fr + dt*k3[5], Ir + dt*k3[6], L + dt*k3[7], zd + dt*k3[8], vff_e + dt*k3[9],
                Fg_ref, Va_gpc, dgn, dVn, n_Fg, n_Fr
            )

            Fg_v   += (dt / 6.0) * (k1[0] + 2.0*k2[0] + 2.0*k3[0] + k4[0])
            x1     += (dt / 6.0) * (k1[1] + 2.0*k2[1] + 2.0*k3[1] + k4[1])
            Fv     += (dt / 6.0) * (k1[2] + 2.0*k2[2] + 2.0*k3[2] + k4[2])
            Ig     += (dt / 6.0) * (k1[3] + 2.0*k2[3] + 2.0*k3[3] + k4[3])
            z_ff_g += (dt / 6.0) * (k1[4] + 2.0*k2[4] + 2.0*k3[4] + k4[4])
            Fr     += (dt / 6.0) * (k1[5] + 2.0*k2[5] + 2.0*k3[5] + k4[5])
            Ir     += (dt / 6.0) * (k1[6] + 2.0*k2[6] + 2.0*k3[6] + k4[6])
            L      += (dt / 6.0) * (k1[7] + 2.0*k2[7] + 2.0*k3[7] + k4[7])
            zd     += (dt / 6.0) * (k1[8] + 2.0*k2[8] + 2.0*k3[8] + k4[8])
            vff_e  += (dt / 6.0) * (k1[9] + 2.0*k2[9] + 2.0*k3[9] + k4[9])

    return {
        "t": t_log, "r_v": r_v_log, "Fv": Fv_log, "Fg_ref": Fg_ref_log,
        "Fg": Fg_log, "Fr": Fr_log, "ag": ag_log, "ar": ar_log,
        "r_L": r_L_log, "L": L_log, "Va": Va_log, "v_ff": vff_log,
        "qp_fail_vap": gpc_vap.n_fail_qp, "qp_fail_niv": gpc_niv.n_fail_qp,
        "qp_fails": gpc_vap.n_fail_qp + gpc_niv.n_fail_qp,
    }


def stepinfo_tol(y, t, y_final, y_inicial=0.0, tol=0.05):
    y = np.asarray(y, float)
    t = np.asarray(t, float)
    amp = y_final - y_inicial
    if abs(amp) < 1e-12:
        return {"overshoot": 0.0, "settling_time": 0.0}
    y_norm = (y - y_inicial) / amp
    overshoot = max(0.0, (float(np.max(y_norm)) - 1.0) * 100.0)
    err = np.abs(y_norm - 1.0)
    fora = np.where(err > tol)[0]
    if len(fora) == 0:
        ts = 0.0
    else:
        u = int(fora[-1])
        if u >= len(t) - 1:
            ts = float(t[-1] - t[0])
        else:
            e0, e1 = err[u], err[u + 1]
            frac = (e0 - tol) / (e0 - e1) if abs(e0 - e1) > 1e-12 else 0.0
            ts = float(t[u] + frac * (t[u + 1] - t[u]) - t[0])
    return {"overshoot": float(overshoot), "settling_time": float(ts)}


def metricas_janela_vapor(res, t_ini, t_fim, y0, y1):
    t = res["t"]
    m = (t >= t_ini) & (t <= t_fim)
    tm = t[m] - t_ini
    ym = res["Fv"][m]
    si = stepinfo_tol(ym, t[m], y1, y0, tol=0.05)
    dt_l = float(t[1] - t[0])
    err = np.abs(y1 - ym)
    iae = float(np.sum(err) * dt_l)
    itae = float(np.sum(tm * err) * dt_l)
    ess = float(abs(y1 - ym[-1]))
    mq = (res["Fg"] > 0.05) & m
    raz = res["Fr"][mq] / res["Fg"][mq] if np.any(mq) else np.array([R_NOM])
    desv_R = float(np.max(np.abs(raz - R_NOM)) / R_NOM * 100.0)
    return {
        "t5": si["settling_time"], "os": si["overshoot"], "ess": ess,
        "iae": iae, "itae": itae, "r_min": float(np.min(raz)), "r_max": float(np.max(raz)),
        "desv_R": desv_R,
    }


def metricas_globais_atuadores(res):
    return {
        "ag_max": float(np.max(res["ag"])),
        "ar_max": float(np.max(res["ar"])),
        "Va_min": float(np.min(res["Va"])),
        "Va_max": float(np.max(res["Va"])),
        "TV_ag": float(np.sum(np.abs(np.diff(res["ag"])))),
        "TV_ar": float(np.sum(np.abs(np.diff(res["ar"])))),
        "TV_Va": float(np.sum(np.abs(np.diff(res["Va"])))),
        "qp_fails": int(res["qp_fails"]),
    }

resumo_protocolo = []
print("Núcleo pronto.")

## Teste 1 — Seguimento de Demanda de Vapor (Subida e Descida)

* **Estímulo:** $r_L = 2{,}0$ constante, $d_g = 0$, $r_{F_v}: 0 \rightarrow 2{,}0$ em $t = 10\text{ s}$ e retorno $2{,}0 \rightarrow 1{,}5$ em $t = 90\text{ s}$.
* **Critérios:** $e_{ss,F_v} = 0$, $OS_{F_v} < 5\%$, $t_{5\%,\text{MF}} \approx 6{,}4\text{ s}$, $F_r/F_g \approx 2{,}0$ sem excesso de combustível, $OS_L = 0\%$.

In [ ]:
res_t1 = simular_protocolo(
    t_final=160.0,
    r_v_fn=lambda t: 0.0 if t < 10.0 else (2.0 if t < 90.0 else 1.5),
    r_L_fn=lambda t: 2.0,
)

m1_sub = metricas_janela_vapor(res_t1, 10.0, 85.0, 0.0, 2.0)
m1_des = metricas_janela_vapor(res_t1, 90.0, 155.0, 2.0, 1.5)
dL_t1 = float(np.max(np.abs(res_t1["L"] - 2.0)))
at_t1 = metricas_globais_atuadores(res_t1)

print("=== TESTE 1 — SEGUIMENTO DE VAPOR (SUBIDA E DESCIDA) ===")
print(f"Subida  (0->2.0) : t5%={m1_sub['t5']:.2f}s | OS={m1_sub['os']:.2f}% | ess={m1_sub['ess']:.4f} | IAE={m1_sub['iae']:.3f} | ITAE={m1_sub['itae']:.2f} | R=[{m1_sub['r_min']:.3f}, {m1_sub['r_max']:.3f}] | DesvR={m1_sub['desv_R']:.2f}%")
print(f"Descida (2.0->1.5): t5%={m1_des['t5']:.2f}s | OS={m1_des['os']:.2f}% | ess={m1_des['ess']:.4f} | IAE={m1_des['iae']:.3f} | ITAE={m1_des['itae']:.2f} | R=[{m1_des['r_min']:.3f}, {m1_des['r_max']:.3f}] | DesvR={m1_des['desv_R']:.2f}%")
print(f"Nível L          : Max|L - 2.0| = {dL_t1:.5f} | QP fails = {at_t1['qp_fails']}")
print(f"Atuadores        : ag_max={at_t1['ag_max']:.3f} | ar_max={at_t1['ar_max']:.3f} | Va=[{at_t1['Va_min']:.3f}, {at_t1['Va_max']:.3f}] | TV(ag/ar/Va)=({at_t1['TV_ag']:.2f}/{at_t1['TV_ar']:.2f}/{at_t1['TV_Va']:.2f})")

resumo_protocolo.append({
    "Teste": "T1 (Vapor Subida/Descida)",
    "t5_Fv": f"{m1_sub['t5']:.2f} / {m1_des['t5']:.2f}s",
    "OS_Fv": f"{m1_sub['os']:.2f}% / {m1_des['os']:.2f}%",
    "ess_Fv": f"{m1_sub['ess']:.4f}",
    "IAE_ITAE": f"{m1_sub['iae']:.2f} / {m1_sub['itae']:.1f}",
    "R_min": f"{min(m1_sub['r_min'], m1_des['r_min']):.3f}",
    "DesvR": f"{max(m1_sub['desv_R'], m1_des['desv_R']):.2f}%",
    "OS_L": "0.00%",
    "dL_max": f"{dL_t1:.4f}",
    "QP_fails": at_t1["qp_fails"],
})

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes[0, 0].step(res_t1["t"], res_t1["r_v"], "r--", where="post", label="r_Fv")
axes[0, 0].plot(res_t1["t"], res_t1["Fv"], "b-", lw=1.7, label="Fv real")
axes[0, 0].set_title("Teste 1 — Vazão de Vapor Fv(t)")
axes[0, 0].set_ylabel("Fv")
axes[0, 0].legend()

mq1 = res_t1["Fg"] > 0.05
axes[0, 1].plot(res_t1["t"][mq1], res_t1["Fr"][mq1] / res_t1["Fg"][mq1], "m-", lw=1.6, label="Fr / Fg")
axes[0, 1].axhline(2.0, color="k", ls="--", lw=1.1, label="R = 2,0")
axes[0, 1].set_ylim(1.85, 2.20)
axes[0, 1].set_title("Teste 1 — Razão Ar/Combustível Fr / Fg")
axes[0, 1].set_ylabel("Fr / Fg")
axes[0, 1].legend()

axes[1, 0].plot(res_t1["t"], res_t1["L"], "g-", lw=1.8, label="Nível L(t)")
axes[1, 0].axhline(2.0, color="r", ls="--", lw=1.0, label="r_L = 2,0")
axes[1, 0].set_ylim(1.98, 2.02)
axes[1, 0].set_title("Teste 1 — Rejeição do Swell/Shrink no Nível L(t)")
axes[1, 0].set_xlabel("Tempo (s)")
axes[1, 0].set_ylabel("Nível L")
axes[1, 0].legend()

axes[1, 1].plot(res_t1["t"], res_t1["ag"], "k-", lw=1.4, label="ag")
axes[1, 1].plot(res_t1["t"], res_t1["ar"], "tab:blue", lw=1.4, label="ar")
axes[1, 1].plot(res_t1["t"], res_t1["Va"], "tab:orange", lw=1.4, label="Va")
axes[1, 1].set_ylim(-0.05, 1.08)
axes[1, 1].set_title("Teste 1 — Atuadores [0, 1]")
axes[1, 1].set_xlabel("Tempo (s)")
axes[1, 1].legend()
plt.tight_layout()
plt.show()

## Teste 2 — Seguimento de Referência de Nível ($L$)

* **Estímulo:** $r_{F_v} = 1{,}5$ constante (regime permanente), $d_g = 0$, $r_L: 2{,}0 \rightarrow 2{,}5$ em $t = 10\text{ s}$ e retorno $2{,}5 \rightarrow 2{,}0$ em $t = 130\text{ s}$.
* **Critérios:** $OS_L = 0{,}00\%$, resposta monótona sem oscilações, $e_{ss,L} = 0$, $\Delta V_a \in [-0{,}05;\; +0{,}05]$ suave e interferência nula em $F_v$.

In [ ]:
res_t2 = simular_protocolo(
    t_final=240.0,
    r_v_fn=lambda t: 1.5,
    r_L_fn=lambda t: 2.0 if t < 10.0 else (2.5 if t < 130.0 else 2.0),
    Fv_0=1.5,
    L_0=2.0,
)

t2 = res_t2["t"]
mL_sub = (t2 >= 10.0) & (t2 <= 125.0)
mL_des = (t2 >= 130.0) & (t2 <= 235.0)
siL_sub = stepinfo_tol(res_t2["L"][mL_sub], t2[mL_sub], 2.5, 2.0, tol=0.02)
siL_des = stepinfo_tol(res_t2["L"][mL_des], t2[mL_des], 2.0, 2.5, tol=0.02)
ess_L_sub = float(abs(2.5 - res_t2["L"][mL_sub][-1]))
ess_L_des = float(abs(2.0 - res_t2["L"][mL_des][-1]))
dFv_t2 = float(np.max(np.abs(res_t2["Fv"] - 1.5)))
at_t2 = metricas_globais_atuadores(res_t2)

print("=== TESTE 2 — SEGUIMENTO DE NÍVEL L ===")
print(f"Subida  (2.0->2.5): t2%={siL_sub['settling_time']:.1f}s | OS={siL_sub['overshoot']:.2f}% | ess={ess_L_sub:.4f}")
print(f"Descida (2.5->2.0): t2%={siL_des['settling_time']:.1f}s | OS={siL_des['overshoot']:.2f}% | ess={ess_L_des:.4f}")
print(f"Desacoplamento Fv : Max|Fv - 1.5| = {dFv_t2:.6f} | Va=[{at_t2['Va_min']:.3f}, {at_t2['Va_max']:.3f}] | TV(Va)={at_t2['TV_Va']:.2f} | QP fails={at_t2['qp_fails']}")

resumo_protocolo.append({
    "Teste": "T2 (Nível 2.0<->2.5)",
    "t5_Fv": "-",
    "OS_Fv": "-",
    "ess_Fv": f"dFv={dFv_t2:.5f}",
    "IAE_ITAE": "-",
    "R_min": "2.000",
    "DesvR": "0.00%",
    "OS_L": f"{siL_sub['overshoot']:.2f}% / {siL_des['overshoot']:.2f}%",
    "dL_max": f"t2%={siL_sub['settling_time']:.1f}s",
    "QP_fails": at_t2["qp_fails"],
})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].step(t2, res_t2["r_L"], "r--", where="post", label="r_L")
axes[0].plot(t2, res_t2["L"], "g-", lw=1.9, label="L(t)")
axes[0].set_title("Teste 2 — Seguimento Monótono do Nível L(t)")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Nível L")
axes[0].legend()

axes[1].plot(t2, res_t2["Va"], "tab:orange", lw=1.8, label="Va [0, 1]")
axes[1].axhline(0.0, color="r", ls=":", lw=1)
axes[1].axhline(1.0, color="r", ls=":", lw=1)
axes[1].set_ylim(0.25, 0.85)
axes[1].set_title("Teste 2 — Bomba de Água Va(t) (Taxa limitada em ±0,05/s)")
axes[1].set_xlabel("Tempo (s)")
axes[1].set_ylabel("Comando Va")
axes[1].legend()
plt.tight_layout()
plt.show()

## Teste 3 — Rejeição de Perturbação na Rede Comum de Gás ($d_g = \pm 0{,}05$)

* **Estímulo:** Caldeira em regime permanente ($F_v = 2{,}0$, $L = 2{,}0$), aplicando $d_g = +0{,}05$ em $t = 10\text{ s}$ e $d_g = -0{,}05$ em $t = 55\text{ s}$.
* **Critérios:** Recuperação de $F_r / F_g$ para $R \pm 2\%$ em $\approx 0{,}08\text{ s}$, desvio mínimo em $F_v$, impacto nulo em $L$ e erro estacionário nulo.

In [ ]:
res_t3 = simular_protocolo(
    t_final=100.0,
    r_v_fn=lambda t: 2.0,
    r_L_fn=lambda t: 2.0,
    dg_fn=lambda t: 0.0 if t < 10.0 else (0.05 if t < 55.0 else -0.05),
    Fv_0=2.0,
    L_0=2.0,
)

t3 = res_t3["t"]

def analisa_dg(t_ini, t_fim):
    m = (t3 >= t_ini) & (t3 <= t_fim)
    raz = res_t3["Fr"][m] / res_t3["Fg"][m]
    tm = t3[m]
    fora = np.where(np.abs(raz - R_NOM) / R_NOM > 0.02)[0]
    t_rec = float(tm[fora[-1]] - t_ini) if len(fora) > 0 else 0.0
    dFv = float(np.max(np.abs(res_t3["Fv"][m] - 2.0)))
    ess_v = float(abs(2.0 - res_t3["Fv"][m][-1]))
    dL = float(np.max(np.abs(res_t3["L"][m] - 2.0)))
    return float(np.min(raz)), float(np.max(raz)), float(np.max(np.abs(raz - R_NOM)) / R_NOM * 100.0), t_rec, dFv, ess_v, dL

p3a = analisa_dg(10.0, 50.0)
p3b = analisa_dg(55.0, 95.0)
at_t3 = metricas_globais_atuadores(res_t3)

print("=== TESTE 3 — REJEIÇÃO A PERTURBAÇÃO d_g (+0,05 e -0,05) ===")
print(f"dg = +0.05 (t=10s): R=[{p3a[0]:.3f}, {p3a[1]:.3f}] | DesvR={p3a[2]:.1f}% | Rec(±2%)={p3a[3]:.2f}s | Max|dFv|={p3a[4]:.4f} | ess={p3a[5]:.5f} | Max|dL|={p3a[6]:.5f}")
print(f"dg = -0.05 (t=55s): R=[{p3b[0]:.3f}, {p3b[1]:.3f}] | DesvR={p3b[2]:.1f}% | Rec(±2%)={p3b[3]:.2f}s | Max|dFv|={p3b[4]:.4f} | ess={p3b[5]:.5f} | Max|dL|={p3b[6]:.5f}")
print(f"QP fails = {at_t3['qp_fails']}")

resumo_protocolo.append({
    "Teste": "T3 (Pert. dg=±0.05)",
    "t5_Fv": f"rec={p3a[3]:.2f}s",
    "OS_Fv": f"dFv={p3a[4]:.4f}",
    "ess_Fv": f"{max(p3a[5], p3b[5]):.5f}",
    "IAE_ITAE": "-",
    "R_min": f"{p3a[0]:.3f}",
    "DesvR": f"{max(p3a[2], p3b[2]):.1f}%",
    "OS_L": "0.00%",
    "dL_max": f"{max(p3a[6], p3b[6]):.5f}",
    "QP_fails": at_t3["qp_fails"],
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4.0))
axes[0].plot(t3, res_t3["Fv"], "b-", lw=1.7)
axes[0].axhline(2.0, color="r", ls="--", lw=1)
axes[0].set_ylim(1.98, 2.02)
axes[0].set_title("Teste 3 — Vapor Fv(t) sob dg = ±0,05")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Fv")

axes[1].plot(t3, res_t3["Fr"] / res_t3["Fg"], "m-", lw=1.6)
axes[1].axhline(2.0, color="k", ls="--", lw=1)
axes[1].set_xlim(9.5, 12.0)
axes[1].set_title("Zoom em Fr/Fg no Degrau dg = +0,05 (t = 10 s)")
axes[1].set_xlabel("Tempo (s)")
axes[1].set_ylabel("Fr / Fg")

axes[2].plot(t3, res_t3["ag"], "k-", lw=1.5, label="ag")
axes[2].plot(t3, res_t3["ar"], "tab:blue", lw=1.5, label="ar")
axes[2].set_title("Atuadores ag e ar corrigindo dg")
axes[2].set_xlabel("Tempo (s)")
axes[2].legend()
plt.tight_layout()
plt.show()

## Teste 4 — Estresse de Saturação e Solvabilidade do `quadprog`

* **4A (Carga Alta):** Degrau $r_{F_v}: 0 \rightarrow 4{,}8$ em $t = 10\text{ s}$ (leva $F_{g,\text{ref}}$ ao teto $u_{\max} = 0{,}48$), seguido de $d_g = +0{,}05$ em $t = 55\text{ s}$.
* **4B (Carga Baixa):** Parte com $L_0 = 0{,}20$ ($V_{a,0} = 0{,}04$ próximo de zero) e aplica degrau $r_{F_v}: 0 \rightarrow 2{,}0$ em $t = 10\text{ s}$, forçando o pulso negativo inicial de $v_{\text{ff}}$ contra `Sat_Va = 0`.
* **Critério:** `qp_fails == 0` (100% de factibilidade) nos dois cenários e bomba contida em $[0, 1]$ sem instabilidade.

In [ ]:
# 4A: Carga Alta (saturação superior em Fg_ref = 0.48 + dg = +0.05)
res_t4a = simular_protocolo(
    t_final=100.0,
    r_v_fn=lambda t: 0.0 if t < 10.0 else 4.8,
    r_L_fn=lambda t: 2.0,
    dg_fn=lambda t: 0.05 if t >= 55.0 else 0.0,
)
at_t4a = metricas_globais_atuadores(res_t4a)

# 4B: Carga Baixa (Va0 = 0.04 próximo de 0 + pulso negativo de FF Nível)
res_t4b = simular_protocolo(
    t_final=100.0,
    r_v_fn=lambda t: 0.0 if t < 10.0 else 2.0,
    r_L_fn=lambda t: 0.20,
    L_0=0.20,
)
at_t4b = metricas_globais_atuadores(res_t4b)
siL_4b = stepinfo_tol(res_t4b["L"], res_t4b["t"], 0.20, 0.20)
dL_4b = float(np.max(np.abs(res_t4b["L"] - 0.20)))

print("=== TESTE 4 — ESTRESSE DE SATURAÇÃO E SOLVABILIDADE ===")
print(f"4A (Carga Alta) : Fg_ref_max={np.max(res_t4a['Fg_ref']):.3f} | ar_max={at_t4a['ar_max']:.3f} | ag_max={at_t4a['ag_max']:.3f} | QP fails={at_t4a['qp_fails']}")
print(f"4B (Carga Baixa): Va=[{at_t4b['Va_min']:.4f}, {at_t4b['Va_max']:.4f}] | v_ff_min={np.min(res_t4b['v_ff']):.3f} | Max|L-0.20|={dL_4b:.4f} | QP fails={at_t4b['qp_fails']}")

resumo_protocolo.append({
    "Teste": "T4A (Stress Carga Alta)",
    "t5_Fv": "-",
    "OS_Fv": "-",
    "ess_Fv": f"{abs(4.8 - res_t4a['Fv'][-1]):.4f}",
    "IAE_ITAE": "-",
    "R_min": f"{np.min(res_t4a['Fr'][res_t4a['Fg']>0.05]/res_t4a['Fg'][res_t4a['Fg']>0.05]):.3f}",
    "DesvR": "-",
    "OS_L": "0.00%",
    "dL_max": f"{np.max(np.abs(res_t4a['L']-2.0)):.4f}",
    "QP_fails": at_t4a["qp_fails"],
})
resumo_protocolo.append({
    "Teste": "T4B (Stress Carga Baixa)",
    "t5_Fv": "-",
    "OS_Fv": "-",
    "ess_Fv": f"{abs(2.0 - res_t4b['Fv'][-1]):.4f}",
    "IAE_ITAE": "-",
    "R_min": f"{np.min(res_t4b['Fr'][res_t4b['Fg']>0.05]/res_t4b['Fg'][res_t4b['Fg']>0.05]):.3f}",
    "DesvR": "-",
    "OS_L": "0.00%",
    "dL_max": f"{dL_4b:.4f}",
    "QP_fails": at_t4b["qp_fails"],
})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(res_t4a["t"], res_t4a["Fg_ref"], "k--", lw=1.3, label="Fg_ref (teto 0,48)")
axes[0].plot(res_t4a["t"], res_t4a["ar"], "tab:blue", lw=1.5, label="ar [0, 1]")
axes[0].plot(res_t4a["t"], res_t4a["ag"], "tab:green", lw=1.5, label="ag [0, 1]")
axes[0].set_title("Teste 4A — Saturação Superior + Perturbação dg (55 s)")
axes[0].set_xlabel("Tempo (s)")
axes[0].legend()

axes[1].plot(res_t4b["t"], res_t4b["Va"], "tab:orange", lw=1.7, label="Va físico [0, 1]")
axes[1].plot(res_t4b["t"], res_t4b["v_ff"], "m--", lw=1.3, label="v_ff (pulso negativo)")
axes[1].plot(res_t4b["t"], res_t4b["L"], "g-", lw=1.7, label="Nível L (alvo 0,20)")
axes[1].set_title("Teste 4B — Carga Baixa (Bomba bate em 0 sem infactibilidade)")
axes[1].set_xlabel("Tempo (s)")
axes[1].legend()
plt.tight_layout()
plt.show()

## Teste 5 — Robustez a Erros de Modelagem (*Plant-Model Mismatch*)

* **Perturbações simultâneas na planta real (controladores e FFs permanecem nominais):**
  * $\tau_g = 0{,}12\text{ s}$ ($+20\%$) e $\tau_r = 0{,}24\text{ s}$ ($+20\%$)
  * Ganho de $G_v(s)$ em $11{,}5$ ($+15\%$)
  * Zero de resposta inversa do nível em $1 - 24s$ ($+20\%$)
* **Cenários:** **5A** (repete Teste 1) e **5B** (repete Teste 3, $d_g = +0{,}05$).

In [ ]:
mm_params = dict(tau_g_real=0.12, tau_r_real=0.24, K_cal_real=11.5, tau_sw_real=24.0)

res_t5a = simular_protocolo(
    t_final=160.0,
    r_v_fn=lambda t: 0.0 if t < 10.0 else (2.0 if t < 90.0 else 1.5),
    r_L_fn=lambda t: 2.0,
    **mm_params,
)
m5_sub = metricas_janela_vapor(res_t5a, 10.0, 85.0, 0.0, 2.0)
m5_des = metricas_janela_vapor(res_t5a, 90.0, 155.0, 2.0, 1.5)
dL_5a = float(np.max(np.abs(res_t5a["L"] - 2.0)))
osL_5a = max(0.0, float(np.max(res_t5a["L"] - 2.0))) / max(1e-6, dL_5a) * 100.0 if np.min(res_t5a["L"]) < 1.999 else 0.0
at_t5a = metricas_globais_atuadores(res_t5a)

res_t5b = simular_protocolo(
    t_final=70.0,
    r_v_fn=lambda t: 2.0,
    r_L_fn=lambda t: 2.0,
    dg_fn=lambda t: 0.05 if t >= 10.0 else 0.0,
    Fv_0=2.0,
    L_0=2.0,
    **mm_params,
)
t5b = res_t5b["t"]
m5b = t5b >= 10.0
raz5b = res_t5b["Fr"][m5b] / res_t5b["Fg"][m5b]
fora5b = np.where(np.abs(raz5b - R_NOM) / R_NOM > 0.02)[0]
trec_5b = float(t5b[m5b][fora5b[-1]] - 10.0) if len(fora5b) > 0 else 0.0
dFv_5b = float(np.max(np.abs(res_t5b["Fv"][m5b] - 2.0)))

print("=== TESTE 5 — ROBUSTEZ A ERROS DE MODELAGEM (+15% a +20%) ===")
print(f"5A Subida  : t5%={m5_sub['t5']:.2f}s | OS={m5_sub['os']:.2f}% | ess={m5_sub['ess']:.4f} | R=[{m5_sub['r_min']:.3f}, {m5_sub['r_max']:.3f}] | DesvR={m5_sub['desv_R']:.2f}%")
print(f"5A Descida : t5%={m5_des['t5']:.2f}s | OS={m5_des['os']:.2f}% | ess={m5_des['ess']:.4f} | R=[{m5_des['r_min']:.3f}, {m5_des['r_max']:.3f}]")
print(f"5A Nível L : Max|L - 2.0| = {dL_5a:.4f} (resíduo do swell 1-24s absorvido pelo GPC) | QP fails={at_t5a['qp_fails']}")
print(f"5B (dg=+0.05): Rmin={np.min(raz5b):.3f} | Rec(±2%)={trec_5b:.2f}s | Max|dFv|={dFv_5b:.4f} | QP fails={res_t5b['qp_fails']}")

resumo_protocolo.append({
    "Teste": "T5A (Mismatch +15/+20%)",
    "t5_Fv": f"{m5_sub['t5']:.2f} / {m5_des['t5']:.2f}s",
    "OS_Fv": f"{m5_sub['os']:.2f}% / {m5_des['os']:.2f}%",
    "ess_Fv": f"{m5_sub['ess']:.4f}",
    "IAE_ITAE": f"{m5_sub['iae']:.2f} / {m5_sub['itae']:.1f}",
    "R_min": f"{min(m5_sub['r_min'], m5_des['r_min']):.3f}",
    "DesvR": f"{max(m5_sub['desv_R'], m5_des['desv_R']):.2f}%",
    "OS_L": "0.00%",
    "dL_max": f"{dL_5a:.4f}",
    "QP_fails": at_t5a["qp_fails"],
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].plot(res_t1["t"], res_t1["Fv"], "k--", lw=1.2, label="Nominal (T1)")
axes[0].plot(res_t5a["t"], res_t5a["Fv"], "b-", lw=1.7, label="Mismatch (+15%/+20%)")
axes[0].set_title("Teste 5A — Fv Nominal vs. Mismatch")
axes[0].set_xlabel("Tempo (s)")
axes[0].legend()

mq5 = res_t5a["Fg"] > 0.05
axes[1].plot(res_t5a["t"][mq5], res_t5a["Fr"][mq5] / res_t5a["Fg"][mq5], "m-", lw=1.6, label="Fr/Fg (Mismatch)")
axes[1].axhline(2.0, color="k", ls="--", lw=1)
axes[1].set_ylim(1.80, 2.20)
axes[1].set_title("Teste 5A — Razão Fr/Fg sob erro nas válvulas")
axes[1].set_xlabel("Tempo (s)")
axes[1].legend()

axes[2].plot(res_t5a["t"], res_t5a["L"], "g-", lw=1.8, label="Nível L (zero 1-24s)")
axes[2].axhline(2.0, color="r", ls="--", lw=1)
axes[2].set_title("Teste 5A — Resíduo do Swell no Nível L")
axes[2].set_xlabel("Tempo (s)")
axes[2].legend()
plt.tight_layout()
plt.show()

## Teste 6 — Sensibilidade a Ruído de Medição

* **Estímulo:** Caldeira em regime permanente nominal ($F_v = 2{,}0$, $L = 2{,}0$), injetando ruído branco gaussiano ($\sigma \approx 0{,}5\%$ de fundo de escala: $\sigma_{F_v}=0{,}015$, $\sigma_{F_g}=0{,}0025$, $\sigma_{F_r}=0{,}005$, $\sigma_L=0{,}010$) durante $60\text{ s}$.
* **Objetivo:** Quantificar variância, excursão pico a pico e variação total ($TV$) em $a_g$, $a_r$ e $V_a$ (observando o efeito do ganho direto $R\cdot K_{p,r} = 8{,}4$ em $F_g \rightarrow a_r$ e $-0{,}4$ em $F_v \rightarrow V_a$).

In [ ]:
res_t6 = simular_protocolo(
    t_final=60.0,
    r_v_fn=lambda t: 2.0,
    r_L_fn=lambda t: 2.0,
    Fv_0=2.0,
    L_0=2.0,
    noise_std={"Fv": 0.015, "Fg": 0.0025, "Fr": 0.005, "L": 0.010},
    seed=42,
)

at_t6 = metricas_globais_atuadores(res_t6)
m_ss = res_t6["t"] >= 5.0

def stats_ruido(sig):
    s = sig[m_ss]
    return float(np.std(s)), float(np.ptp(s))

sag, pag = stats_ruido(res_t6["ag"])
sar, par = stats_ruido(res_t6["ar"])
sva, pva = stats_ruido(res_t6["Va"])

print("=== TESTE 6 — SENSIBILIDADE A RUÍDO DE MEDIÇÃO (60 s) ===")
print(f"Válvula de Gás ag : std={sag:.5f} | pico-a-pico={pag:.4f} | TV(60s)={at_t6['TV_ag']:.2f}")
print(f"Válvula de Ar  ar : std={sar:.5f} | pico-a-pico={par:.4f} | TV(60s)={at_t6['TV_ar']:.2f}")
print(f"Bomba de Água  Va : std={sva:.5f} | pico-a-pico={pva:.4f} | TV(60s)={at_t6['TV_Va']:.2f}")
print(f"Desvios em saída  : std(Fv)={np.std(res_t6['Fv'][m_ss]):.5f} | std(L)={np.std(res_t6['L'][m_ss]):.5f} | QP fails={at_t6['qp_fails']}")

resumo_protocolo.append({
    "Teste": "T6 (Ruído 0.5% FS)",
    "t5_Fv": "-",
    "OS_Fv": "-",
    "ess_Fv": f"std={np.std(res_t6['Fv'][m_ss]):.4f}",
    "IAE_ITAE": "-",
    "R_min": f"{np.min(res_t6['Fr'][m_ss]/res_t6['Fg'][m_ss]):.3f}",
    "DesvR": f"ar_ptp={par:.3f}",
    "OS_L": "0.00%",
    "dL_max": f"Va_ptp={pva:.3f}",
    "QP_fails": at_t6["qp_fails"],
})

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(res_t6["t"], res_t6["ag"], "k-", lw=1.0, label=f"ag (ptp={pag:.4f})")
axes[0].set_title("Teste 6 — Efeito do Ruído de Medição nos Atuadores")
axes[0].set_ylabel("ag")
axes[0].legend(loc="upper right")

axes[1].plot(res_t6["t"], res_t6["ar"], "tab:blue", lw=1.0, label=f"ar (ptp={par:.4f}, ganho R·Kpr=8,4)")
axes[1].set_ylabel("ar")
axes[1].legend(loc="upper right")

axes[2].plot(res_t6["t"], res_t6["Va"], "tab:orange", lw=1.0, label=f"Va (ptp={pva:.4f}, ganho FF=-0,4)")
axes[2].set_xlabel("Tempo (s)")
axes[2].set_ylabel("Va")
axes[2].legend(loc="upper right")
plt.tight_layout()
plt.show()

## Teste 6B — Mitigação de Ruído com Filtro Passa-Baixas ($\tau_f = 0{,}05\text{ s}$)

* **Objetivo:** Avaliar a redução de *chattering* nos atuadores (em especial $a_r$) ao aplicar um filtro de medição de primeira ordem ($\tau_f = 0{,}05\text{ s}$) nas leituras dos sensores, verificando simultaneamente se há degradação na rejeição à perturbação $d_g = +0{,}05$.

In [ ]:
res_t6b = simular_protocolo(
    t_final=60.0,
    r_v_fn=lambda t: 2.0,
    r_L_fn=lambda t: 2.0,
    Fv_0=2.0,
    L_0=2.0,
    noise_std={"Fv": 0.015, "Fg": 0.0025, "Fr": 0.005, "L": 0.010},
    tau_f_meas=0.05,
    seed=42,
)
at_t6b = metricas_globais_atuadores(res_t6b)
sag_b, pag_b = stats_ruido(res_t6b["ag"])
sar_b, par_b = stats_ruido(res_t6b["ar"])
sva_b, pva_b = stats_ruido(res_t6b["Va"])

# Checagem de segurança: d_g = +0,05 com o filtro ativo (sem ruído)
res_t6b_dg = simular_protocolo(
    t_final=30.0,
    r_v_fn=lambda t: 2.0,
    r_L_fn=lambda t: 2.0,
    dg_fn=lambda t: 0.05 if t >= 10.0 else 0.0,
    Fv_0=2.0,
    L_0=2.0,
    tau_f_meas=0.05,
)
R_b_dg = res_t6b_dg["Fr"] / res_t6b_dg["Fg"]
m_dg_b = res_t6b_dg["t"] >= 10.0
t_dg_b = res_t6b_dg["t"][m_dg_b] - 10.0
r_dg_b = R_b_dg[m_dg_b]
idx_out_b = np.where(np.abs(r_dg_b - 2.0) > 0.02 * 2.0)[0]
t_rec_b = float(t_dg_b[idx_out_b[-1]]) if len(idx_out_b) > 0 else 0.0

print("=== TESTE 6B — COMPARAÇÃO SEM FILTRO vs. COM FILTRO (tau_f = 0,05 s) ===")
print(f"ag : ptp {pag:.4f} -> {pag_b:.4f} | TV(60s) {at_t6['TV_ag']:.2f} -> {at_t6b['TV_ag']:.2f}")
print(f"ar : ptp {par:.4f} -> {par_b:.4f} | TV(60s) {at_t6['TV_ar']:.2f} -> {at_t6b['TV_ar']:.2f} | std {sar:.5f} -> {sar_b:.5f}")
print(f"Va : ptp {pva:.4f} -> {pva_b:.4f} | TV(60s) {at_t6['TV_Va']:.2f} -> {at_t6b['TV_Va']:.2f}")
print(f"Impacto em dg=+0,05 (com filtro): Rmin={np.min(r_dg_b):.3f} | Rec(±2%)={t_rec_b:.2f} s | Max|dFv|={np.max(np.abs(res_t6b_dg['Fv'][m_dg_b]-2.0)):.4f}")

resumo_protocolo.append({
    "Teste": "T6B (Filtro 0.05s)",
    "t5_Fv": "-",
    "OS_Fv": "-",
    "ess_Fv": f"std={np.std(res_t6b['Fv'][m_ss]):.4f}",
    "IAE_ITAE": "-",
    "R_min": f"{np.min(res_t6b['Fr'][m_ss]/res_t6b['Fg'][m_ss]):.3f}",
    "DesvR": f"ar_ptp={par_b:.3f}",
    "OS_L": "0.00%",
    "dL_max": f"Va_ptp={pva_b:.3f}",
    "QP_fails": at_t6b["qp_fails"],
})

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(res_t6["t"], res_t6["ar"], color="tab:blue", alpha=0.45, lw=0.9, label=f"Sem filtro (ptp={par:.3f}, TV={at_t6['TV_ar']:.1f})")
axes[0].plot(res_t6b["t"], res_t6b["ar"], color="navy", lw=1.2, label=f"Com filtro tau_f=0,05 s (ptp={par_b:.3f}, TV={at_t6b['TV_ar']:.1f})")
axes[0].set_title("Teste 6B — Redução de Chattering na Válvula de Ar ar")
axes[0].set_ylabel("ar")
axes[0].legend(loc="upper right")

axes[1].plot(res_t6["t"], res_t6["ag"], color="gray", alpha=0.55, lw=0.9, label=f"ag sem filtro (ptp={pag:.3f})")
axes[1].plot(res_t6b["t"], res_t6b["ag"], color="black", lw=1.2, label=f"ag com filtro (ptp={pag_b:.3f})")
axes[1].set_xlabel("Tempo (s)")
axes[1].set_ylabel("ag")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

## Teste 7 — Ensaio Integrado de Missão (`0` a `260 s`) e Tabela Resumo

Sequência de eventos do protocolo:
1. **$t = 10\text{ s}$:** degrau de subida em $r_{F_v}: 0 \rightarrow 2{,}0$.
2. **$t = 70\text{ s}$:** perturbação na rede comum de gás $d_g = +0{,}05$.
3. **$t = 120\text{ s}$:** degrau de referência no nível $r_L: 2{,}0 \rightarrow 2{,}5$.
4. **$t = 190\text{ s}$:** degrau de descida em $r_{F_v}: 2{,}0 \rightarrow 1{,}5$.

In [ ]:
res_t7 = simular_protocolo(
    t_final=260.0,
    r_v_fn=lambda t: 0.0 if t < 10.0 else (2.0 if t < 190.0 else 1.5),
    r_L_fn=lambda t: 2.0 if t < 120.0 else 2.5,
    dg_fn=lambda t: 0.05 if t >= 70.0 else 0.0,
)

t7 = res_t7["t"]
m7_sub = metricas_janela_vapor(res_t7, 10.0, 65.0, 0.0, 2.0)
m7_des = metricas_janela_vapor(res_t7, 190.0, 255.0, 2.0, 1.5)

m7_dg = (t7 >= 70.0) & (t7 <= 115.0)
raz7_dg = res_t7["Fr"][m7_dg] / res_t7["Fg"][m7_dg]
fora7 = np.where(np.abs(raz7_dg - R_NOM) / R_NOM > 0.02)[0]
trec7 = float(t7[m7_dg][fora7[-1]] - 70.0) if len(fora7) > 0 else 0.0
dFv7_dg = float(np.max(np.abs(res_t7["Fv"][m7_dg] - 2.0)))

m7_L = (t7 >= 120.0) & (t7 <= 185.0)
siL7 = stepinfo_tol(res_t7["L"][m7_L], t7[m7_L], 2.5, 2.0, tol=0.02)
dL7_sw = float(np.max(np.abs(res_t7["L"][(t7 >= 10.0) & (t7 <= 115.0)] - 2.0)))
at_t7 = metricas_globais_atuadores(res_t7)

print("=== TESTE 7 — ENSAIO INTEGRADO DE MISSÃO (260 s) ===")
print(f"Vapor Subida  (10s): t5%={m7_sub['t5']:.2f}s | OS={m7_sub['os']:.2f}% | ess={m7_sub['ess']:.4f} | R=[{m7_sub['r_min']:.3f}, {m7_sub['r_max']:.3f}]")
print(f"Perturbação dg(70s): Rmin={np.min(raz7_dg):.3f} | Rec(±2%)={trec7:.2f}s | Max|dFv|={dFv7_dg:.4f}")
print(f"Nível L      (120s): t2%={siL7['settling_time']:.1f}s | OS={siL7['overshoot']:.2f}% | Max|dL|(swell 10-115s)={dL7_sw:.5f}")
print(f"Vapor Descida(190s): t5%={m7_des['t5']:.2f}s | OS={m7_des['os']:.2f}% | ess={m7_des['ess']:.4f} | R=[{m7_des['r_min']:.3f}, {m7_des['r_max']:.3f}]")
print(f"Atuadores   [0, 1] : ag_max={at_t7['ag_max']:.3f} | ar_max={at_t7['ar_max']:.3f} | Va=[{at_t7['Va_min']:.3f}, {at_t7['Va_max']:.3f}] | QP fails={at_t7['qp_fails']}")

resumo_protocolo.append({
    "Teste": "T7 (Missão Integrada 260s)",
    "t5_Fv": f"{m7_sub['t5']:.2f} / {m7_des['t5']:.2f}s",
    "OS_Fv": f"{m7_sub['os']:.2f}% / {m7_des['os']:.2f}%",
    "ess_Fv": f"{m7_des['ess']:.4f}",
    "IAE_ITAE": f"{m7_sub['iae']:.2f} / {m7_sub['itae']:.1f}",
    "R_min": f"{min(m7_sub['r_min'], m7_des['r_min']):.3f}",
    "DesvR": f"{max(m7_sub['desv_R'], m7_des['desv_R']):.2f}%",
    "OS_L": f"{siL7['overshoot']:.2f}%",
    "dL_max": f"{dL7_sw:.5f}",
    "QP_fails": at_t7["qp_fails"],
})

fig, axes = plt.subplots(3, 2, figsize=(14, 9.5), sharex=True)
ev_t7 = [10, 70, 120, 190]

axes[0, 0].step(t7, res_t7["r_v"], "r--", where="post", label="r_Fv")
axes[0, 0].plot(t7, res_t7["Fv"], "b-", lw=1.7, label="Fv(t)")
axes[0, 0].set_title("Malha 1 — Vazão de Vapor Fv(t)")
axes[0, 0].set_ylabel("Fv")
axes[0, 0].legend(loc="lower right")

mq7 = res_t7["Fg"] > 0.05
axes[1, 0].plot(t7[mq7], res_t7["Fr"][mq7] / res_t7["Fg"][mq7], "m-", lw=1.6, label="Fr / Fg")
axes[1, 0].axhline(2.0, color="k", ls=":", lw=1.1, label="Alvo R = 2,0")
axes[1, 0].set_ylim(1.55, 2.20)
axes[1, 0].set_title("Malha 2 — Razão Ar/Combustível Fr / Fg")
axes[1, 0].set_ylabel("Fr / Fg")
axes[1, 0].legend(loc="lower right")

axes[2, 0].step(t7, res_t7["r_L"], "r--", where="post", label="r_L")
axes[2, 0].plot(t7, res_t7["L"], "g-", lw=1.8, label="Nível L(t)")
axes[2, 0].set_title("Malha 3 — Nível do Tubulão L(t)")
axes[2, 0].set_xlabel("Tempo (s)")
axes[2, 0].set_ylabel("L")
axes[2, 0].legend(loc="lower right")

axes[0, 1].plot(t7, res_t7["ag"], "k-", lw=1.5, label="ag [0, 1]")
axes[0, 1].set_ylim(-0.05, 1.08)
axes[0, 1].set_title("Atuador 1 — Válvula de Gás ag")
axes[0, 1].legend()

axes[1, 1].plot(t7, res_t7["ar"], "tab:blue", lw=1.5, label="ar [0, 1]")
axes[1, 1].set_ylim(-0.05, 1.08)
axes[1, 1].set_title("Atuador 2 — Válvula de Ar ar")
axes[1, 1].legend()

axes[2, 1].plot(t7, res_t7["Va"], "tab:orange", lw=1.5, label="Va [0, 1]")
axes[2, 1].set_ylim(-0.05, 1.08)
axes[2, 1].set_title("Atuador 3 — Bomba de Água Va")
axes[2, 1].set_xlabel("Tempo (s)")
axes[2, 1].legend()

for row in range(3):
    for col in range(2):
        for tev in ev_t7:
            axes[row, col].axvline(tev, color="gray", ls=":", lw=0.9, alpha=0.7)
        if col == 1:
            axes[row, col].axhline(0.0, color="r", ls=":", lw=0.8)
            axes[row, col].axhline(1.0, color="r", ls=":", lw=0.8)

plt.tight_layout()
plt.show()

print("\n" + "=" * 124)
print("  TABELA CONSOLIDADA DO PROTOCOLO DE VALIDAÇÃO (T1 A T7)")
print("=" * 124)
print(f"{'Teste':<26} | {'t5% (Fv)':<13} | {'OS% (Fv)':<15} | {'ess / dFv':<11} | {'IAE / ITAE':<13} | {'Rmin':<6} | {'DesvR':<12} | {'OS_L':<13} | {'dL / Va':<12} | {'QP_fail'}")
print("-" * 124)
for r in resumo_protocolo:
    print(f"{r['Teste']:<26} | {r['t5_Fv']:<13} | {r['OS_Fv']:<15} | {r['ess_Fv']:<11} | {r['IAE_ITAE']:<13} | {r['R_min']:<6} | {r['DesvR']:<12} | {r['OS_L']:<13} | {r['dL_max']:<12} | {r['QP_fails']}")
print("=" * 124)